# 03. SmoothQuant MXFP4

本章介绍 SmoothQuant 和 MXFP4 的基本原理，并给出基于 AMD Quark `quantize_quark.py` 的 Qwen2.5-7B-Instruct 量化与 vLLM 验证流程。

参考资料：

- AMD Quark SmoothQuant 文档：<https://quark.docs.amd.com/latest/pytorch/smoothquant.html>
- ROCm AI Developer Hub MXFP4 + Quark + vLLM 教程：<https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/notebooks/gpu_dev_optimize/mxfp4_quantization_quark_vllm.html#software>
- AMD Quark GitHub：<https://github.com/amd/Quark>


## 1. SmoothQuant 原理

LLM 的 activation 中可能存在少量显著 outlier。低 bit 量化时，outlier 会扩大量化范围，使多数普通数值的有效量化分辨率下降，从而影响模型精度。

SmoothQuant 的核心思想是：**在不改变线性层数学结果的前提下，将一部分量化难度从 activation 迁移到 weight**。

对线性层，可表示为：

```text
Y = XW
  = (X / s) (sW)
```

其中 `s` 是按 channel 计算得到的平滑系数。`X / s` 用于降低 activation outlier，`sW` 将对应尺度变化补偿到权重中，因此浮点数学结果保持等价；同时，量化时的 activation 分布更平滑，有助于降低低 bit activation 量化误差。

![SmoothQuant 原理图](assets/smoothquant/smoothquant_principle.png)


### `alpha` / migration strength

SmoothQuant 通常会有一个控制迁移强度的参数，常见写法是 `alpha` 或 migration strength。

- `alpha` 越接近 `0`：对 activation 的缩放越弱，activation quantization 承担更多量化压力。
- `alpha` 越接近 `1`：对 activation outlier 的平滑越强，更多量化压力转移到 weight quantization。
- 实验中通常从中间值开始，并结合校准集和评估集观察困惑度、任务精度和生成质量。


## 2. MXFP4 原理

MXFP4 即 **Microscaling Floating Point 4**。它使用 4-bit floating point 表示数值，并在固定大小的 block 内共享 scale。ROCm AI Developer Hub 的 MXFP4 + Quark + vLLM 教程中介绍了使用 Quark 将模型量化为 MXFP4，并通过 vLLM 在 AMD GPU 上推理的流程。下图以常见的 32 个元素共享一个 8-bit scale 为例说明 block scaling 机制。

```text
一组原始浮点值
  -> 按 block 分组
  -> 每个 block 计算共享 scale
  -> block 内每个元素用 4-bit FP 表示
  -> 推理时结合共享 scale 近似恢复数值范围
```

![MXFP4 block scaling 示意图](assets/smoothquant/mxfp4_block_scaling.png)


## 3. 本章量化流程

本章使用 `quantize_quark.py` 完成量化，主线聚焦命令行流程。

```text
准备模型目录
  -> 进入 Quark LLM PTQ 示例目录
  -> 使用 mxfp4 + smoothquant 导出 Hugging Face 格式模型
  -> 使用 vLLM --quantization quark 做 chat smoke test
```

主示例参数：

| 参数 | 值 |
| --- | --- |
| 模型 | `Qwen2.5-7B-Instruct` |
| quant scheme | `mxfp4` |
| quant algo | `smoothquant` |
| dataset | `pileval` |
| calibration samples | `8` |
| seq len | `128` |
| batch size | `1` |
| export format | `hf_format` |


### `quantize_quark.py` 工作流程细节

`quantize_quark.py` 会先解析 CLI 参数，再加载模型、tokenizer 和校准数据；随后根据模型类型从 `LLMTemplate` 构建 `quant_config`，并由 `ModelQuantizer` 执行量化、冻结和导出。本章命令使用 `--model_export hf_format` 导出 Hugging Face 格式模型，并使用 `--skip_evaluation` 跳过脚本内置评估，后续直接进入 vLLM smoke test。

![quantize_quark.py 工作流程图](assets/smoothquant/quantize_quark_workflow.png)


## 4. 准备脚本环境

下面拉取 Quark 示例代码，并切换到对应的 `examples/torch/language_modeling/llm_ptq` 目录。大文件建议放置在 `/root` 下。


运行下面的 cell 安装系统工具和 Python 依赖，下载并解压 AMD Quark 示例代码，同时安装与教程版本匹配的 `amd-quark==0.11.1`。


In [ ]:
# 安装基础包
!sudo apt update
!sudo apt install unzip -y
!pip install 'huggingface-hub<1.0' evaluate accelerate datasets pillow transformers zstandard lm-eval==0.4.9.2

# 下载和解压 AMD Quark 示例（包括配置文件和用例脚本）
!wget -O amd_quark-0.11.1.zip https://download.amd.com/opendownload/Quark/amd_quark-0.11.1.zip
!unzip -o amd_quark-0.11.1.zip

# 从 PyPI 安装 AMD Quark Python 包
!pip install amd-quark==0.11.1

运行下面的 cell 设置 `REPO_PATH` 环境变量并进入 Quark LLM PTQ 示例目录。后续命令会在该目录下调用 `quantize_quark.py`，同时通过 `REPO_PATH` 访问本教程仓库中的辅助脚本。


In [ ]:
# 设置好 REPO_PATH，用于访问教程脚本
import os

os.environ["REPO_PATH"] = os.path.dirname(os.getcwd())
print(os.environ["REPO_PATH"])

# 进入工作目录

%cd ./amd_quark-0.11.1/examples/torch/language_modeling/llm_ptq

# 安装 llm_ptq 所需依赖，查看 quantize_quark.py 参数
!pip install -r requirements.txt
!python quantize_quark.py --help

运行下面的 cell 确认 `REPO_PATH` 已正确设置。输出应指向本教程仓库根目录。


In [ ]:
!echo $REPO_PATH

## 5. 准备模型

本章示例使用 `Qwen2.5-7B-Instruct`。如果 Hugging Face 访问受限，可根据环境需要配置 `HF_ENDPOINT`。


运行下面的 cell 下载 `Qwen2.5-7B-Instruct` 到 `/root/Qwen2.5-7B-Instruct`。如果环境可直接访问 Hugging Face，可移除 `HF_ENDPOINT`。


In [ ]:
!HF_ENDPOINT=https://hf-mirror.com huggingface-cli download \
    --resume-download Qwen/Qwen2.5-7B-Instruct \
    --local-dir /root/Qwen2.5-7B-Instruct


## 6. mxfp4 + SmoothQuant 量化导出

以下命令使用 `pileval` 作为校准数据集，按 `mxfp4` scheme 启用 SmoothQuant，并导出 Hugging Face 格式模型。示例使用较小校准规模，便于快速验证流程；正式评估时可根据显存、时间和精度要求调整 `--num_calib_data` 与 `--seq_len`。


运行下面的 cell 启动 MXFP4 + SmoothQuant 量化。该命令使用 `pileval` 做少量校准，并将 Hugging Face 格式模型导出到 `/root/outputs/qwen25_7b_mxfp4_smoothquant_hf`。


In [ ]:
!HF_ENDPOINT=https://hf-mirror.com python quantize_quark.py \
    --model_dir /root/Qwen2.5-7B-Instruct \
    --output_dir /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
    --quant_scheme mxfp4 \
    --quant_algo smoothquant \
    --model_export hf_format \
    --dataset pileval \
    --num_calib_data 8 \
    --seq_len 128 \
    --batch_size 1 \
    --skip_evaluation

## 7. 可选：mxfp4 baseline

如果需要评估 SmoothQuant 的影响，可以保留相同的模型、校准数据、序列长度和 batch size，仅去掉 `--quant_algo smoothquant` 生成 `mxfp4` baseline。


运行下面的 cell 生成不启用 SmoothQuant 的 MXFP4 baseline。保持模型、校准数据和导出格式一致，便于后续对比 SmoothQuant 的影响。


In [ ]:
!HF_ENDPOINT=https://hf-mirror.com python quantize_quark.py \
    --model_dir /root/Qwen2.5-7B-Instruct \
    --output_dir /root/outputs/qwen25_7b_mxfp4_hf \
    --quant_scheme mxfp4 \
    --model_export hf_format \
    --dataset pileval \
    --num_calib_data 8 \
    --seq_len 128 \
    --batch_size 1 \
    --skip_evaluation
!pwd

## 8. vLLM chat smoke test

量化导出完成后，先用 vLLM 做一次轻量级生成验证。该步骤用于确认导出的 Hugging Face 格式模型可以被 vLLM 以 `--quantization quark` 方式加载，并能够完成一次短文本生成。

本教程仓库提供 `course_scripts/vllm_chat_smoke.py`。会实际调用 vLLM API 进行 inference，下面命令使用 `$REPO_PATH/course_scripts/vllm_chat_smoke.py`，如果实际存放路径不同，请相应调整 `python` 后面的脚本路径。

下面命令验证 `mxfp4 + SmoothQuant` 输出目录。若要验证 baseline，将 `--model-dir` 改为 `/root/outputs/qwen25_7b_mxfp4_hf`，并相应调整 `--output-json` 文件名。


运行下面的 cell 使用 vLLM 加载 SmoothQuant 量化模型，并完成一次短文本生成验证。该步骤用于确认导出的 Quark 格式模型可以通过 `--quantization quark` 正常推理。


In [ ]:
!python $REPO_PATH/course_scripts/vllm_chat_smoke.py \
    --model-dir /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
    --quantization quark \
    --dtype float16 \
    --enforce-eager \
    --max-model-len 1024 \
    --gpu-memory-utilization 0.85 \
    --max-tokens 48 \
    --use-chat-template \
    --prompt "你好，请用一句话解释模型量化。" \
    --output-json /root/run_logs/qwen25_7b_mxfp4_smoothquant_vllm_smoke_quantization_prompt.json


运行下面的 cell 做一次轻量级 vLLM latency bench。这里使用较短输入输出长度和较少迭代次数，适合快速确认推理链路是否可用。


In [ ]:
# vllm bench测试
# !vllm bench latency --model /root/outputs/qwen25_7b_mxfp4_smoothquant_hf --tokenizer /root/outputs/qwen25_7b_mxfp4_smoothquant_hf --input-len 4096 --output-len 1024 --tensor-parallel 1 

# 快速验证
!vllm bench latency \
  --model /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
  --tokenizer /root/outputs/qwen25_7b_mxfp4_smoothquant_hf \
  --input-len 32 \
  --output-len 32 \
  --num-iters-warmup 1 \
  --num-iters 3


## 9. 记录结果

建议记录以下字段，便于后续对比 SmoothQuant、baseline 或其他量化配置：

| 字段 | 示例 |
| --- | --- |
| 模型 | `Qwen2.5-7B-Instruct` |
| quant scheme | `mxfp4` |
| quant algo | `smoothquant` / 无 |
| calibration dataset | `pileval` |
| calibration samples | `8` |
| seq len | `128` |
| batch size | `1` |
| export format | `hf_format` |
| SmoothQuant output | `/root/outputs/qwen25_7b_mxfp4_smoothquant_hf` |
| baseline output | `/root/outputs/qwen25_7b_mxfp4_hf` |
| smoke test result | `qwen25_7b_mxfp4_smoothquant_vllm_smoke_quantization_prompt.json` |


## 10. 本章小结

- SmoothQuant 使用等价缩放降低 activation outlier，并将部分量化压力迁移到 weight。
- MXFP4 使用 microscaling 方式，将数值按 block 分组，并为每个 block 维护共享 scale。
- 本章命令统一使用 `--quant_scheme mxfp4`；通过 `--quant_algo smoothquant` 启用 SmoothQuant。
- `quantize_quark.py` 负责模型校准、量化和导出；vLLM smoke test 用于验证导出模型能否被 `--quantization quark` 正常加载。
